# 金融对话数值推理模型：SFT → 轻量 DPO / GRPO → benchmark 评估

目标：训练一个面向 **金融对话数值推理** 的 reasoning model。

整体闭环：
- **SFT-1 主干推理**：`ConvFinQA + FinQA`
- **SFT-2 中文补强**：`fingpt-fineval + 少量 fingpt-fiqa_qa`
- **DPO（可选）**：小规模优化表达质量、结构和少废话
- **GRPO（推荐）**：基于可验证 reward 优化答案正确性、程序一致性与结构约束
- **benchmark 评估**：`FinQA / ConvFinQA + CFLUE + FinanceBench + AdaptLLM/finance-tasks`

本 Notebook 复用 MedicalGPT 的训练框架：
- 数据格式遵循 `docs/datasets.md`
- pipeline 参考 `run_training_dpo_pipeline.ipynb`


## 0. 环境准备（可选）

如果你在全新环境运行，请先安装依赖；已按项目 README 配好可跳过。


In [ ]:
!pip install modelscope


In [14]:
# HF Mirror（可选）
import os
HF_ENDPOINT = "https://hf-mirror.com"
os.environ["HF_ENDPOINT"] = HF_ENDPOINT
print("HF_ENDPOINT:", os.getenv("HF_ENDPOINT"))


HF_ENDPOINT: https://hf-mirror.com


## 1. 任务配置

这里把配置拆成五部分：
1. 主干推理数据 `SFT-1`
2. 中文补强数据 `SFT-2`
3. 原始数据下载缓存目录
4. 转换 / 清洗 / 混合目录
5. 训练与评估输出目录


In [ ]:
from pathlib import Path
import json
import random
from itertools import islice

BASE_MODEL = "/root/autodl-tmp/models/qwen/Qwen2.5-3B-Instruct"
TEMPLATE_NAME = "qwen"
RANDOM_SEED = 42
FORCE_REDOWNLOAD_RAW = False

OUT_DIR = Path("data/financial_reasoning")
RAW_CACHE_DIR = OUT_DIR / "raw"
SFT1_SFT_DIR = OUT_DIR / "sft1_sharegpt"
SFT2_SFT_DIR = OUT_DIR / "sft2_sharegpt"
DPO_DIR = OUT_DIR / "dpo_pairs"
MIXED_DIR = OUT_DIR / "mixed"
CLEAN_DIR = OUT_DIR / "clean"
REPORT_DIR = OUT_DIR / "reports"

SFT1_MIXED_FILE = MIXED_DIR / "train_sft1_mixed.jsonl"
SFT2_MIXED_FILE = MIXED_DIR / "train_sft2_mixed.jsonl"
DPO_MIXED_FILE = DPO_DIR / "train_reasoning_dpo.jsonl"
SFT1_CLEAN_FILE = CLEAN_DIR / "train_sft1_clean.jsonl"
SFT2_CLEAN_FILE = CLEAN_DIR / "train_sft2_clean.jsonl"

SFT1_DIR = CLEAN_DIR / "sft1_dir"
SFT2_DIR = CLEAN_DIR / "sft2_dir"
DPO_TRAIN_DIR = DPO_DIR / "train_dir"

SFT1_OUT = Path("outputs/financial_reasoning_sft1_lora")
SFT1_MERGED_OUT = Path("outputs/financial_reasoning_sft1_merged")
SFT2_OUT = Path("outputs/financial_reasoning_sft2_lora")
SFT2_MERGED_OUT = Path("outputs/financial_reasoning_sft2_merged")
DPO_OUT = Path("outputs/financial_reasoning_dpo_lora")
DPO_MERGED_OUT = Path("outputs/financial_reasoning_dpo_merged")
TB_LOG_DIR = Path("outputs/tensorboard/financial_reasoning")

for d in [RAW_CACHE_DIR, SFT1_SFT_DIR, SFT2_SFT_DIR, DPO_DIR, MIXED_DIR, CLEAN_DIR, REPORT_DIR, SFT1_DIR, SFT2_DIR, DPO_TRAIN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SFT1_TOTAL_BUDGET = 12000
SFT2_EXTRA_BUDGET = 2500
DPO_TOTAL_BUDGET = 4000
MAX_DPO_PER_DATASET = 2000

SFT1_DATA_SPECS = [
    {
        "name": "fingpt_convfinqa_train",
        "family": "convfinqa_turn",
        "source_type": "local",
        "local_path": RAW_CACHE_DIR / "convfinqa_turn" / "train_turn.json",
        "split": "train",
        "weight": 1.0,
        "max_rows": None,
        "description": "多轮金融对话 + 数值推理主干数据（使用本地已上传文件）",
    },
    {
        "name": "finqa_train",
        "family": "finqa",
        "source_type": "local",
        "local_path": RAW_CACHE_DIR / "finqa" / "train.json",
        "split": "train",
        "weight": 0.7,
        "max_rows": 6251,
        "description": "表格 + 文本 + 程序推理主干数据（使用本地已上传文件）",
    },
]

SFT2_DATA_SPECS = [
    {
        "name": "fingpt_fineval_train",
        "family": "fineval",
        "source_type": "hf",
        "dataset_name": "FinGPT/fingpt-fineval",
        "split": "train",
        "weight": 1.0,
        "max_rows": 1060,
        "description": "中文金融考试推理补强",
    },
    {
        "name": "fingpt_fiqa_qa_train",
        "family": "fiqa_qa",
        "source_type": "hf",
        "dataset_name": "FinGPT/fingpt-fiqa_qa",
        "split": "train",
        "weight": 0.35,
        "max_rows": 1500,
        "description": "少量中文/对话式金融 QA 补强",
    },
]

BENCHMARK_SPECS = [
    {"name": "ConvFinQA_dev", "role": "核心多轮数值推理"},
    {"name": "FinQA_dev", "role": "核心表文混合推理"},
    {"name": "CFLUE", "role": "中文金融泛化"},
    {"name": "FinanceBench", "role": "开放书金融 QA 迁移"},
    {"name": "AdaptLLM/finance-tasks", "role": "补充 benchmark"},
]


def allocate_target_rows(specs, total_budget: int, default_cap: int | None = None):
    if not specs:
        return []
    total_weight = sum(max(float(spec.get("weight", 0.0)), 0.0) for spec in specs)
    if total_weight <= 0:
        raise ValueError("All dataset weights are zero; cannot allocate target_rows.")

    allocated = []
    for spec in specs:
        raw_target = int(round(total_budget * float(spec.get("weight", 0.0)) / total_weight))
        cap = spec.get("max_rows", default_cap)
        if cap is not None:
            raw_target = min(raw_target, int(cap))
        spec = dict(spec)
        spec["target_rows"] = max(raw_target, 0)
        allocated.append(spec)
    return allocated


SFT1_DATA_SPECS = allocate_target_rows(SFT1_DATA_SPECS, SFT1_TOTAL_BUDGET)
SFT2_DATA_SPECS = allocate_target_rows(SFT2_DATA_SPECS, SFT2_EXTRA_BUDGET)

print("BASE_MODEL:", BASE_MODEL)
print("SFT1_TOTAL_BUDGET:", SFT1_TOTAL_BUDGET)
print("SFT2_EXTRA_BUDGET:", SFT2_EXTRA_BUDGET)
print("DPO_TOTAL_BUDGET:", DPO_TOTAL_BUDGET)
print("SFT1_DATA_SPECS:", json.dumps([{**spec, 'local_path': str(spec.get('local_path', ''))} for spec in SFT1_DATA_SPECS], ensure_ascii=False, indent=2))
print("SFT2_DATA_SPECS:", json.dumps(SFT2_DATA_SPECS, ensure_ascii=False, indent=2))
print("BENCHMARK_SPECS:", json.dumps(BENCHMARK_SPECS, ensure_ascii=False, indent=2))


BASE_MODEL: /root/autodl-tmp/models/qwen/Qwen2.5-3B-Instruct
SFT1_DATA_SPECS: [
  {
    "name": "convfinqa_train_turn",
    "family": "convfinqa_turn",
    "source_type": "hf",
    "dataset_name": "AdaptLLM/ConvFinQA",
    "config": "train_turn",
    "split": "train",
    "target_rows": 9000,
    "description": "多轮金融对话 + 数值推理主干数据"
  },
  {
    "name": "finqa_train",
    "family": "finqa",
    "source_type": "url_json",
    "dataset_name": "ibm-research/finqa",
    "split": "train",
    "target_rows": 6251,
    "description": "表格 + 文本 + 程序推理主干数据"
  }
]
SFT2_DATA_SPECS: [
  {
    "name": "fingpt_fineval_train",
    "family": "fineval",
    "source_type": "hf",
    "dataset_name": "FinGPT/fingpt-fineval",
    "split": "train",
    "target_rows": 1060,
    "description": "中文金融考试推理补强"
  },
  {
    "name": "fingpt_fiqa_qa_train",
    "family": "fiqa_qa",
    "source_type": "hf",
    "dataset_name": "FinGPT/fingpt-fiqa_qa",
    "split": "train",
    "target_rows": 1500,
    "description": "少量

## 1.1 统一任务格式

推荐训练模板：
- **ConvFinQA**：历史对话 + 当前问题 + 表格/文本上下文 → 分步推理 → 最终答案
- **FinQA**：文本 + 表格 + 问题 → reasoning program / 中间步骤 → 执行结果
- **fingpt-fineval**：题目 + 选项 → 简短推理解释 → 正确选项
- **fingpt-fiqa_qa**：问题 → 结构化解释 → 结论

MedicalGPT 对数据格式的要求：
- **SFT**：`conversations`
- **DPO**：`question + response_chosen + response_rejected`


In [25]:
from pprint import pprint

SFT_SCHEMA_EXAMPLE = {
    "conversations": [
        {"from": "human", "value": "历史对话 + 当前问题 + 文本/表格上下文"},
        {"from": "gpt", "value": "问题分析...\n关键证据...\n推理程序...\n最终答案..."},
    ]
}

DPO_SCHEMA_EXAMPLE = {
    "system": "",
    "history": [],
    "question": "统一格式的金融推理问题",
    "response_chosen": "结构化且正确的回答",
    "response_rejected": "答案错误或程序不一致的回答",
}

print("[MedicalGPT SFT schema]")
pprint(SFT_SCHEMA_EXAMPLE, sort_dicts=False)
print("[MedicalGPT DPO schema]")
pprint(DPO_SCHEMA_EXAMPLE, sort_dicts=False)


[MedicalGPT SFT schema]
{'conversations': [{'from': 'human', 'value': '历史对话 + 当前问题 + 文本/表格上下文'},
                   {'from': 'gpt',
                    'value': '问题分析...\n关键证据...\n推理程序...\n最终答案...'}]}
[MedicalGPT DPO schema]
{'system': '',
 'history': [],
 'question': '统一格式的金融推理问题',
 'response_chosen': '结构化且正确的回答',
 'response_rejected': '答案错误或程序不一致的回答'}


In [26]:
print("[Training stages]")
print({
    "SFT-1": [spec["name"] for spec in SFT1_DATA_SPECS],
    "SFT-2": [spec["name"] for spec in SFT2_DATA_SPECS],
    "DPO": "从上述 raw 数据自动构造轻量偏好对",
    "GRPO": "基于格式奖励 + 程序一致性 + 答案正确性",
})


[Training stages]
{'SFT-1': ['convfinqa_train_turn', 'finqa_train'], 'SFT-2': ['fingpt_fineval_train', 'fingpt_fiqa_qa_train'], 'DPO': '从上述 raw 数据自动构造轻量偏好对', 'GRPO': '基于格式奖励 + 程序一致性 + 答案正确性'}


## 2. 下载原始数据到本地缓存

这一阶段只负责下载 raw 数据，避免每次重跑 notebook 都重复下载。
- `url_json`：直接下载官方 JSON 文件
- `hf`：通过 `datasets.load_dataset` 落地到本地 jsonl
- 若缓存已存在且 `FORCE_REDOWNLOAD_RAW=False`，则直接跳过


In [1]:
import json
from datasets import load_dataset

ALL_DATA_SPECS = SFT1_DATA_SPECS + SFT2_DATA_SPECS


def raw_cache_file(spec: dict) -> Path:
    if spec["source_type"] == "local":
        return Path(spec["local_path"])
    return RAW_CACHE_DIR / spec["family"] / f"{spec['name']}.jsonl"

raw_files = {}
for spec in ALL_DATA_SPECS:
    cache_file = raw_cache_file(spec)
    raw_files[spec["name"]] = cache_file
    cache_file.parent.mkdir(parents=True, exist_ok=True)

    if spec["source_type"] == "local":
        if not cache_file.exists():
            raise FileNotFoundError(f"Missing local raw file: {cache_file}")
        print(f"[use local] {spec['name']} -> {cache_file}")
        continue

    if cache_file.exists() and not FORCE_REDOWNLOAD_RAW:
        print(f"[skip] use cached raw file: {cache_file}")
        continue

    print(f"[download] {spec['name']} -> {cache_file}")
    if spec["source_type"] == "hf":
        ds = load_dataset(spec["dataset_name"], split=spec["split"])
        with cache_file.open('w', encoding='utf-8') as f:
            for row in ds:
                f.write(json.dumps(dict(row), ensure_ascii=False) + "\n")
    else:
        raise ValueError(f"Unsupported source_type: {spec['source_type']}")
    print(f"[saved] {cache_file}")


NameError: name 'SFT1_DATA_SPECS' is not defined

## 2.1 展示原始数据前几条案例

按你的要求，这里直接展示主干数据集 `ConvFinQA / FinQA` 的前几条 raw 样例。

说明：
- `ConvFinQA` 直接读取你已上传的本地文件：`data/financial_reasoning/raw/convfinqa_turn/train_turn.json`
- `FinQA` 直接读取你已上传的本地文件：`data/financial_reasoning/raw/finqa/train.json`


In [ ]:
def show_first_records(path: Path, n: int = 2):
    print(f"\n=== {path} ===")
    with path.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            obj = json.loads(line)
            print(json.dumps(obj, ensure_ascii=False, indent=2)[:3000])

for spec in SFT1_DATA_SPECS:
    show_first_records(raw_files[spec["name"]], n=2)


## 3. 转换格式：raw → MedicalGPT SFT / DPO

这一阶段只读取本地 raw 文件：
- `fin_to_sharegpt.py`：转换为 SFT 格式
- `fin_to_dpo_pairs.py`：转换为轻量 DPO 格式


In [ ]:
import subprocess

sft_reports = []
dpo_reports = []

for spec in ALL_DATA_SPECS:
    raw_file = raw_files[spec["name"]]
    out_dir = SFT1_SFT_DIR if spec in SFT1_DATA_SPECS else SFT2_SFT_DIR
    sft_file = out_dir / f"{spec['name']}_sharegpt.jsonl"
    dpo_file = DPO_DIR / f"{spec['name']}_dpo.jsonl"

    sharegpt_cmd = [
        "python", "fin_to_sharegpt.py",
        "--source_file", str(raw_file),
        "--output_file", str(sft_file),
        "--dataset_family", spec["family"],
    ]
    dpo_cmd = [
        "python", "fin_to_dpo_pairs.py",
        "--source_file", str(raw_file),
        "--output_file", str(dpo_file),
        "--dataset_family", spec["family"],
        "--seed", str(RANDOM_SEED),
    ]

    print(" ".join(sharegpt_cmd))
    sharegpt_result = subprocess.run(sharegpt_cmd, check=True, capture_output=True, text=True)
    print(sharegpt_result.stdout)
    sft_reports.append(json.loads(sharegpt_result.stdout))

    print(" ".join(dpo_cmd))
    dpo_result = subprocess.run(dpo_cmd, check=True, capture_output=True, text=True)
    print(dpo_result.stdout)
    dpo_reports.append(json.loads(dpo_result.stdout))

print("[SFT conversion reports]")
print(json.dumps(sft_reports, ensure_ascii=False, indent=2))
print("[DPO conversion reports]")
print(json.dumps(dpo_reports, ensure_ascii=False, indent=2))


## 3.1 展示转换后的前几条案例

按你的要求，这里展示转换后的 `ConvFinQA / FinQA` SFT 样例，确认模板、表格上下文、历史对话和推理程序是否正常。


In [ ]:
for spec in SFT1_DATA_SPECS:
    sft_file = SFT1_SFT_DIR / f"{spec['name']}_sharegpt.jsonl"
    print(f"\n=== {spec['name']} converted sample ===")
    with sft_file.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 2:
                break
            print(json.dumps(json.loads(line), ensure_ascii=False, indent=2)[:3500])


## 4. 数据集清洗

使用 `clean_sharegpt_dataset.py` 对转换后的 SFT 文件进行统一清洗：
- 去掉空轮次 / 奇数轮次
- 去掉超长样本
- 限制单条样本的上下文长度


In [ ]:
def sample_jsonl_records(path: Path, target_rows: int, seed: int = 42):
    with path.open('r', encoding='utf-8') as f:
        records = [line for line in f if line.strip()]
    if target_rows <= 0 or len(records) <= target_rows:
        return records
    rng = random.Random(seed)
    idxs = list(range(len(records)))
    rng.shuffle(idxs)
    idxs = sorted(idxs[:target_rows])
    return [records[i] for i in idxs]

with SFT1_MIXED_FILE.open('w', encoding='utf-8') as wf:
    for spec in SFT1_DATA_SPECS:
        path = SFT1_SFT_DIR / f"{spec['name']}_sharegpt.jsonl"
        sampled = sample_jsonl_records(path, spec['target_rows'], seed=RANDOM_SEED)
        for line in sampled:
            wf.write(line if line.endswith('\n') else line + '\n')

with SFT2_MIXED_FILE.open('w', encoding='utf-8') as wf:
    for line in sample_jsonl_records(SFT1_MIXED_FILE, 10**9, seed=RANDOM_SEED):
        wf.write(line if line.endswith('\n') else line + '\n')
    for spec in SFT2_DATA_SPECS:
        path = SFT2_SFT_DIR / f"{spec['name']}_sharegpt.jsonl"
        sampled = sample_jsonl_records(path, spec['target_rows'], seed=RANDOM_SEED)
        for line in sampled:
            wf.write(line if line.endswith('\n') else line + '\n')

for src, dst in [(SFT1_MIXED_FILE, SFT1_CLEAN_FILE), (SFT2_MIXED_FILE, SFT2_CLEAN_FILE)]:
    clean_cmd = [
        'python', 'clean_sharegpt_dataset.py',
        '--source_file', str(src),
        '--output_file', str(dst),
        '--min_turns', '2',
        '--max_turns', '16',
        '--max_total_chars', '6000',
        '--max_single_value_chars', '2500',
    ]
    print(' '.join(clean_cmd))
    result = subprocess.run(clean_cmd, check=True, capture_output=True, text=True)
    print(result.stdout)

# Make train_file_dir-style folders.
(SFT1_DIR / SFT1_CLEAN_FILE.name).write_text(SFT1_CLEAN_FILE.read_text(encoding='utf-8'), encoding='utf-8')
(SFT2_DIR / SFT2_CLEAN_FILE.name).write_text(SFT2_CLEAN_FILE.read_text(encoding='utf-8'), encoding='utf-8')


## 5. 数据集配比说明

当前改为“按比例自动计算”：
- 不再手写固定 `target_rows`
- 先设定每个阶段的**总预算**，再设定每个数据集的**权重 weight**
- 最终 `target_rows = total_budget × weight / sum(weights)`，再受 `max_rows` 上限约束

当前默认策略：
- **SFT-1 总预算**：`SFT1_TOTAL_BUDGET = 12000`
- **SFT-2 额外补强预算**：`SFT2_EXTRA_BUDGET = 2500`
- **DPO 总预算**：`DPO_TOTAL_BUDGET = 4000`
- **单数据集 DPO 上限**：`MAX_DPO_PER_DATASET = 2000`

当前权重：
- `fingpt-convfinqa : FinQA = 1.0 : 0.7`
- `fineval : fiqa_qa = 1.0 : 0.35`

这样做的好处：
- 数据规模调整时，不需要手动重写每个 `target_rows`
- 主干数据和补强数据的比例更透明
- 小数据集会自动受 `max_rows` 限制，不会被分配超过全量的样本数


In [ ]:
def line_count(path: Path) -> int:
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())

report = {
    'sft1_total_budget': SFT1_TOTAL_BUDGET,
    'sft2_extra_budget': SFT2_EXTRA_BUDGET,
    'dpo_total_budget': DPO_TOTAL_BUDGET,
    'sft1_allocations': [
        {
            'name': spec['name'],
            'weight': spec['weight'],
            'max_rows': spec['max_rows'],
            'target_rows': spec['target_rows'],
        }
        for spec in SFT1_DATA_SPECS
    ],
    'sft2_allocations': [
        {
            'name': spec['name'],
            'weight': spec['weight'],
            'max_rows': spec['max_rows'],
            'target_rows': spec['target_rows'],
        }
        for spec in SFT2_DATA_SPECS
    ],
    'sft1_raw_mix_rows': line_count(SFT1_MIXED_FILE),
    'sft1_clean_rows': line_count(SFT1_CLEAN_FILE),
    'sft2_raw_mix_rows': line_count(SFT2_MIXED_FILE),
    'sft2_clean_rows': line_count(SFT2_CLEAN_FILE),
}
print(json.dumps(report, ensure_ascii=False, indent=2))


## 6. SFT-1：主干推理训练

对应你的 TODO：
1. 数据集来源改为 `ConvFinQA & FinQA`
2. notebook 提供展示前几条案例的单元
3. 已清洗数据
4. 已统一转换为 MedicalGPT SFT 格式
5. 这里给出 `SFT-1` 训练命令


In [ ]:
sft1_cmd = [
    'python', 'supervised_finetuning.py',
    '--model_name_or_path', BASE_MODEL,
    '--tokenizer_name_or_path', BASE_MODEL,
    '--train_file_dir', str(SFT1_DIR),
    '--validation_split_percentage', '1',
    '--do_train',
    '--use_peft',
    '--num_train_epochs', '3',
    '--per_device_train_batch_size', '1',
    '--gradient_accumulation_steps', '16',
    '--gradient_checkpointing', 'True',
    '--warmup_ratio', '0.05',
    '--weight_decay', '0.05',
    '--learning_rate', '2e-5',
    '--logging_steps', '10',
    '--save_steps', '200',
    '--report_to', 'tensorboard',
    '--logging_dir', str(TB_LOG_DIR / 'sft1'),
    '--model_max_length', '1024',
    '--target_modules', 'all',
    '--lora_rank', '8',
    '--lora_alpha', '16',
    '--lora_dropout', '0.05',
    '--torch_dtype', 'float16',
    '--device_map', 'auto',
    '--output_dir', str(SFT1_OUT),
    '--template_name', TEMPLATE_NAME,
]
print(' '.join(sft1_cmd))
# subprocess.run(sft1_cmd, check=True)


## 7. SFT-2：中文补强

在 `SFT-1` 完成后：
- 先 merge `SFT-1 LoRA`
- 再用 `fingpt-fineval + 少量 fiqa_qa` 做二阶段 SFT


In [ ]:
merge_sft1_cmd = [
    'python', 'merge_peft_adapter.py',
    '--base_model', BASE_MODEL,
    '--tokenizer_path', BASE_MODEL,
    '--lora_model', str(SFT1_OUT),
    '--output_dir', str(SFT1_MERGED_OUT),
]
print(' '.join(merge_sft1_cmd))
# subprocess.run(merge_sft1_cmd, check=True)

sft2_cmd = [
    'python', 'supervised_finetuning.py',
    '--model_name_or_path', str(SFT1_MERGED_OUT),
    '--tokenizer_name_or_path', BASE_MODEL,
    '--train_file_dir', str(SFT2_DIR),
    '--validation_split_percentage', '1',
    '--do_train',
    '--use_peft',
    '--num_train_epochs', '2',
    '--per_device_train_batch_size', '1',
    '--gradient_accumulation_steps', '16',
    '--gradient_checkpointing', 'True',
    '--learning_rate', '1e-5',
    '--logging_steps', '10',
    '--save_steps', '200',
    '--report_to', 'tensorboard',
    '--logging_dir', str(TB_LOG_DIR / 'sft2'),
    '--model_max_length', '1024',
    '--target_modules', 'all',
    '--lora_rank', '8',
    '--lora_alpha', '16',
    '--lora_dropout', '0.05',
    '--torch_dtype', 'float16',
    '--device_map', 'auto',
    '--output_dir', str(SFT2_OUT),
    '--template_name', TEMPLATE_NAME,
]
print(' '.join(sft2_cmd))
# subprocess.run(sft2_cmd, check=True)


## 8. 轻量 DPO（可选）

轻量 DPO 目标：
- 优化表达质量
- 保持结构完整
- 减少废话
- 不让偏好训练覆盖主干 reasoning 能力


In [ ]:
with DPO_MIXED_FILE.open('w', encoding='utf-8') as wf:
    all_specs = SFT1_DATA_SPECS + SFT2_DATA_SPECS
    total_source_target = sum(spec['target_rows'] for spec in all_specs if spec['target_rows'] > 0)
    for spec in all_specs:
        path = DPO_DIR / f"{spec['name']}_dpo.jsonl"
        if total_source_target > 0:
            proportional_target = int(round(DPO_TOTAL_BUDGET * spec['target_rows'] / total_source_target))
        else:
            proportional_target = 0
        dpo_target = min(MAX_DPO_PER_DATASET, spec.get('max_rows') or 10**9, proportional_target)
        sampled = sample_jsonl_records(path, dpo_target, seed=RANDOM_SEED)
        for line in sampled:
            wf.write(line if line.endswith('
') else line + '
')

(DPO_TRAIN_DIR / DPO_MIXED_FILE.name).write_text(DPO_MIXED_FILE.read_text(encoding='utf-8'), encoding='utf-8')

merge_sft2_cmd = [
    'python', 'merge_peft_adapter.py',
    '--base_model', BASE_MODEL,
    '--tokenizer_path', BASE_MODEL,
    '--lora_model', str(SFT2_OUT),
    '--output_dir', str(SFT2_MERGED_OUT),
]
print(' '.join(merge_sft2_cmd))
# subprocess.run(merge_sft2_cmd, check=True)

dpo_cmd = [
    'python', 'dpo_training.py',
    '--model_name_or_path', str(SFT2_MERGED_OUT),
    '--tokenizer_name_or_path', BASE_MODEL,
    '--template_name', TEMPLATE_NAME,
    '--train_file_dir', str(DPO_TRAIN_DIR),
    '--validation_split_percentage', '1',
    '--do_train',
    '--use_peft', 'True',
    '--per_device_train_batch_size', '1',
    '--gradient_accumulation_steps', '16',
    '--gradient_checkpointing', 'True',
    '--learning_rate', '5e-7',
    '--max_steps', '200',
    '--max_source_length', '1024',
    '--max_target_length', '512',
    '--logging_steps', '10',
    '--save_steps', '200',
    '--target_modules', 'all',
    '--lora_rank', '8',
    '--lora_alpha', '16',
    '--lora_dropout', '0.05',
    '--torch_dtype', 'float16',
    '--device_map', 'auto',
    '--output_dir', str(DPO_OUT),
]
print(' '.join(dpo_cmd))
# subprocess.run(dpo_cmd, check=True)


## 9. GRPO reward 设计（推荐）

针对金融 reasoning，推荐 3 类 reward：
1. **格式奖励**：是否满足结构化输出格式
2. **程序一致性奖励**：是否包含与 gold program 一致的关键操作
3. **答案正确性奖励**：最终答案是否与 gold answer 一致 / 数值接近

这里给出 reward 设计原型，后续可以把它接到 `grpo_training.py`。


In [ ]:
import re

SECTION_PATTERNS = {
    'analysis': r'问题分析：',
    'program': r'推理程序：',
    'answer': r'最终答案：',
}


def reasoning_format_reward(text: str) -> float:
    score = 0.0
    for pattern in SECTION_PATTERNS.values():
        if re.search(pattern, text):
            score += 1.0
    return score / len(SECTION_PATTERNS)


def extract_final_answer(text: str) -> str:
    m = re.search(r'最终答案：\s*(.+)', text)
    return m.group(1).strip() if m else text.strip()


def normalize_number(text: str):
    m = re.search(r'-?\d+(?:,\d{3})*(?:\.\d+)?', text.replace(',', ''))
    return float(m.group(0)) if m else None


def answer_correctness_reward(pred: str, gold: str, tol: float = 1e-4) -> float:
    pred_num = normalize_number(extract_final_answer(pred))
    gold_num = normalize_number(gold)
    if pred_num is not None and gold_num is not None:
        return 1.0 if abs(pred_num - gold_num) <= tol else 0.0
    return 1.0 if extract_final_answer(pred).strip() == gold.strip() else 0.0


def program_consistency_reward(pred: str, gold_program: str) -> float:
    if not gold_program:
        return 0.0
    pred_program = ''
    m = re.search(r'推理程序：\s*(.+)', pred)
    if m:
        pred_program = m.group(1).strip()
    gold_ops = [op for op in ['add', 'subtract', 'multiply', 'divide', 'greater', 'table_max', 'table_min'] if op in gold_program]
    if not gold_ops:
        return 0.0
    hit = sum(1 for op in gold_ops if op in pred_program)
    return hit / len(gold_ops)

print('reward design ready')


## 10. benchmark 评估计划

评估闭环：
- **FinQA / ConvFinQA**：核心数值推理能力
- **CFLUE**：中文金融泛化能力
- **FinanceBench**：开放书金融 QA 迁移能力
- **AdaptLLM/finance-tasks**：补充 reasoning / QA 评测


In [ ]:
EVAL_PLAN = {
    'core_reasoning': ['ConvFinQA_dev', 'FinQA_dev'],
    'zh_generalization': ['CFLUE'],
    'open_book_transfer': ['FinanceBench'],
    'supplementary': ['AdaptLLM/finance-tasks'],
}
print(json.dumps(EVAL_PLAN, ensure_ascii=False, indent=2))
